# Task 3: Course Recommendation System

**Primary user: students.** OULAD is student learning data (profiles, enrolments,
clickstream) and contains nothing about staff, so the use case the data actually
supports is *recommending a student's next course to support their progression*.
We say so explicitly rather than forcing a staff use case the data can't back.

This notebook covers every requirement in the brief:

| # | Requirement | Where |
|---|-------------|-------|
| 1 | Two approaches + comparison | Method A (content-based) & Method B (collaborative filtering) |
| 2 | Define + justify similarity metric | "Similarity metric" section |
| 3 | Cold-start, implemented | "Cold start" section |
| 4 | Evaluate with a metric | "Evaluation" section (hit-rate@k + coverage) |
| 5 | Which user, and why | stated above |


In [ ]:
import numpy as np
import pandas as pd

# --- CONFIG ---
DATA_DIR = "../datasets"   # notebook lives in task 3/, data sits in datasets/
K = 3                       # recommend 3 courses

## 0. Load the data
We only need four tables: enrolments + student profiles (`studentInfo`), and the
course metadata (`courses`, `vle`, `assessments`).

In [ ]:
courses     = pd.read_csv(f"{DATA_DIR}/courses.csv")
vle         = pd.read_csv(f"{DATA_DIR}/vle.csv")
assessments = pd.read_csv(f"{DATA_DIR}/assessments.csv")
students    = pd.read_csv(f"{DATA_DIR}/studentInfo.csv")

MODULES = sorted(students["code_module"].unique())
M = len(MODULES)
m_index = {m: i for i, m in enumerate(MODULES)}
print(f"{M} courses: {MODULES}")

## 1. Interactions: who took what
Each student can appear in several module-presentations, so we collapse them to
the **set of distinct modules** a student has ever taken, then build a binary
`students x courses` matrix `B` (1 = took it). This single matrix powers
collaborative filtering and the evaluation.

In [ ]:
student_modules = (students.groupby("id_student")["code_module"]
                   .apply(lambda s: set(s.unique())).to_dict())
STUDENT_IDS = sorted(student_modules.keys())
s_index = {s: i for i, s in enumerate(STUDENT_IDS)}

B = np.zeros((len(STUDENT_IDS), M), dtype=float)
for sid, mods in student_modules.items():
    for mod in mods:
        B[s_index[sid], m_index[mod]] = 1.0

multi = sum(len(v) >= 2 for v in student_modules.values())
print(f"{len(STUDENT_IDS)} students; {multi} took 2+ courses "
      f"(these drive collaborative filtering and evaluation)")

## Similarity metric (and why)
We use **cosine similarity** throughout.

- It is **scale-invariant**: it compares the *direction* of two vectors, not
  their magnitude, so a student who took many courses isn't treated as
  "bigger" than one who took few.
- It behaves well on the **sparse, high-dimensional** vectors we have here.
- It works for **both** representations: binary co-enrolment vectors (Method B)
  and continuous feature vectors (Method A), so the two methods stay comparable.

**Why not Jaccard?** Jaccard similarity, computed as students who took *both* courses over students who took *either*, is a natural fit for the binary "took it / didn't"
data in collaborative filtering, and on that half alone it would be perfectly
valid. We did **not** use it because it only works on binary set data: it cannot
handle the *continuous* feature vectors (course length, activity mix, academic
profile) used in the content-based method. Choosing Jaccard would force two
different similarity metrics across the two methods and make comparing them less
clean. Cosine works for both, so we use it throughout.

**Why not Euclidean distance?** It is dominated by magnitude, so popular courses
and highly-active students look "far" from everything simply because they are
large, which is the wrong bias for recommendation.

In [ ]:
def l2_normalize(X, axis=1, eps=1e-9):
    n = np.linalg.norm(X, axis=axis, keepdims=True)
    return X / np.maximum(n, eps)

def cosine_matrix(X):
    Xn = l2_normalize(X, axis=1)
    return Xn @ Xn.T

def _rank(scores, exclude, k):
    """Top-k items by score, skipping anything in `exclude` (already taken)."""
    order = np.argsort(-scores)
    recs = [MODULES[i] for i in order if MODULES[i] not in exclude]
    return recs[:k]

## 2. Course metadata features (for Method A)
Method A describes each course with two blocks of features:

1. **Intrinsic metadata**: module length, the *mix* of VLE activity types
   (forum-heavy vs quiz-heavy vs reading-heavy), and the *mix* of assessment
   types (TMA / CMA / Exam).
2. **Academic profile of its students**: the average education level, age band,
   deprivation band, prior attempts, credits, and disability rate of the people
   who take it. This is what lets us match a course to a *student's* profile.

Both blocks are z-scored so no single feature dominates the distance.

In [ ]:
# block 1a: module length
length = (courses.groupby("code_module")["module_presentation_length"]
          .mean().reindex(MODULES))

# block 1b: activity-type mix (proportion of sites of each type)
act = (vle.groupby(["code_module", "activity_type"]).size()
       .unstack(fill_value=0).reindex(MODULES).fillna(0))
act = act.div(act.sum(axis=1).replace(0, 1), axis=0)
act.columns = [f"act_{c}" for c in act.columns]

# block 1c: assessment-type mix (count of each type)
asm = (assessments.groupby(["code_module", "assessment_type"]).size()
       .unstack(fill_value=0).reindex(MODULES).fillna(0))
asm.columns = [f"assess_{c}" for c in asm.columns]

meta = pd.concat([length.rename("length"), act, asm], axis=1).fillna(0)

In [ ]:
# academic encoders shared by students AND courses
EDUC_MAP = {"No Formal quals": 0, "Lower Than A Level": 1,
            "A Level or Equivalent": 2, "HE Qualification": 3,
            "Post Graduate Qualification": 4}
AGE_MAP = {"0-35": 0, "35-55": 1, "55<=": 2}
def imd_to_num(x):
    try:    return float(str(x).split("-")[0].replace("%", ""))
    except: return np.nan

stu = students.copy()
stu["educ_n"]  = stu["highest_education"].map(EDUC_MAP)
stu["age_n"]   = stu["age_band"].map(AGE_MAP)
stu["imd_n"]   = stu["imd_band"].map(imd_to_num)
stu["disab_n"] = (stu["disability"] == "Y").astype(float)
ACAD_COLS = ["educ_n","age_n","imd_n","disab_n","num_of_prev_attempts","studied_credits"]

# one academic profile per student, z-scored on the student population
stu_acad = stu.groupby("id_student")[ACAD_COLS].mean().reindex(STUDENT_IDS)
acad_mean, acad_std = stu_acad.mean(), stu_acad.std().replace(0, 1)
stu_acad = stu_acad.fillna(acad_mean)
stu_acad_z = (stu_acad - acad_mean) / acad_std

# course academic block = mean academic of its students, SAME z-scoring
course_acad_raw = (stu.groupby("code_module")[ACAD_COLS].mean()
                   .reindex(MODULES).fillna(acad_mean))
course_acad_z = (course_acad_raw - acad_mean) / acad_std

# z-score intrinsic metadata across the 7 courses
meta_mean, meta_std = meta.mean(), meta.std().replace(0, 1)
meta_z = (meta - meta_mean) / meta_std

# full course vector = [intrinsic metadata | academic profile]
COURSE_VEC = np.hstack([meta_z.values, course_acad_z.values])
META_DIM = meta_z.shape[1]
print("course feature vector length:", COURSE_VEC.shape[1])

## Method A: Content-based
We build a student vector in the **same space** as the courses:

- **metadata block** = the average metadata of the courses they've already taken
  (their "taste"); zeros if they have no history.
- **academic block** = the student's *own* academic profile.

We then recommend the unseen courses whose vector is most cosine-similar to the
student's. Because the academic block needs no history, **this method degrades
gracefully to cold-start**, its key advantage over Method B.

In [ ]:
def content_student_vector(history_mods, academic_z_row):
    if history_mods:
        idx = [m_index[m] for m in history_mods]
        meta_part = meta_z.values[idx].mean(axis=0)
    else:
        meta_part = np.zeros(META_DIM)        # cold student: academic drives it
    return np.hstack([meta_part, academic_z_row])

def content_recommend(sid, k=K):
    hist = student_modules.get(sid, set())
    v = content_student_vector(hist, stu_acad_z.loc[sid].values).reshape(1, -1)
    sims = (l2_normalize(v) @ l2_normalize(COURSE_VEC).T).ravel()
    return _rank(sims, hist, k)

## Method B: Collaborative filtering (item-based)
Ignore course content entirely; learn only from behaviour. We compute a
**course-to-course** cosine similarity from the columns of `B` ("how much do two
courses share an audience"). To recommend for a student, we sum each candidate
course's similarity to the courses they've already taken, then take the top-k
unseen. New students have no history, so we route them to cold-start.

In [ ]:
ITEM_SIM = cosine_matrix(B.T)        # M x M
np.fill_diagonal(ITEM_SIM, 0.0)      # a course can't recommend itself

# Inspect which courses share an audience (higher = more co-enrolled):
print("Course-course similarity:")
print(pd.DataFrame(np.round(ITEM_SIM, 2), index=MODULES, columns=MODULES))

def cf_recommend(sid, k=K):
    hist = student_modules.get(sid, set())
    if not hist:
        return cold_start_recommend(sid, k)
    scores = B[s_index[sid]] @ ITEM_SIM
    return _rank(scores, hist, k)

## 3. Cold start
A brand-new student has no enrolment history, so Method B's scores are all zero.
But the **sign-up form still gives us their academic profile**. So our strategy:

1. **Profile known** → run Method A with an empty history. The academic block
   alone finds "courses suited to students like you." *(implemented)*
2. **Nothing known at all** → fall back to **global popularity** (most-enrolled
   courses). *(implemented)*

This is why content-based isn't just an alternative to CF; it's also our
cold-start engine.

In [ ]:
POPULARITY = B.sum(axis=0)

def popularity_recommend(k=K, exclude=set()):
    return _rank(POPULARITY.astype(float), exclude, k)

def cold_start_recommend(sid_or_profile, k=K):
    if isinstance(sid_or_profile, (int, np.integer)) and sid_or_profile in s_index:
        acad = stu_acad_z.loc[sid_or_profile].values
    elif isinstance(sid_or_profile, dict):     # raw profile of a brand-new student
        row = pd.Series(sid_or_profile)
        acad = ((row[ACAD_COLS] - acad_mean) / acad_std).fillna(0).values
    else:
        return popularity_recommend(k)         # nothing known -> popularity
    v = content_student_vector(set(), acad).reshape(1, -1)
    sims = (l2_normalize(v) @ l2_normalize(COURSE_VEC).T).ravel()
    return _rank(sims, set(), k)

## 4. Evaluation: leave-one-out hit-rate@k + coverage
For every student with 2+ courses we **hide one course**, ask the recommender
for its top-k, and check whether the hidden course comes back (**hit-rate@k**).
We also report **coverage**: the fraction of the 7 courses that ever get
recommended, a diversity check.

> ⚠️ With only 7 courses and k=3, even random guessing scores ~0.5, so the raw
> hit-rate isn't impressive on its own. The meaningful signal is whether a method
> **beats the popularity baseline**, which is what shows the personalisation adds
> value.

In [ ]:
def evaluate(recommender, k=K):
    # Full leave-one-out: for each student with 2+ courses, hold out EACH course
    # in turn. Deterministic (we sort) and uses more of the signal.
    hits = total = 0; recommended = set()
    for sid, mods in student_modules.items():
        if len(mods) < 2: continue
        mods_sorted = sorted(mods)
        for held in mods_sorted:
            kept = set(mods_sorted) - {held}
            saved, saved_row = student_modules[sid], B[s_index[sid]].copy()
            student_modules[sid] = kept
            B[s_index[sid]] = 0.0
            for m in kept: B[s_index[sid], m_index[m]] = 1.0
            try:
                recs = recommender(sid, k)
            finally:                              # always restore state
                student_modules[sid], B[s_index[sid]] = saved, saved_row
            hits += int(held in recs); total += 1; recommended.update(recs)
    return {"hit_rate@k": round(hits/total, 3) if total else float("nan"),
            "coverage": round(len(recommended)/M, 3), "n_eval": total}

print("content-based :", evaluate(content_recommend))
print("collaborative :", evaluate(cf_recommend))
print("popularity    :", evaluate(lambda s, k=K: popularity_recommend(k, student_modules[s])))

## Worked examples & cold-start demo
Sanity-check the recommendations on real students, plus a brand-new student we
only have a profile for.

In [ ]:
for sid in [s for s in STUDENT_IDS if len(student_modules[s]) >= 2][:3]:
    print(f"Student {sid} took {sorted(student_modules[sid])}")
    print(f"   content-based -> {content_recommend(sid)}")
    print(f"   collaborative -> {cf_recommend(sid)}")

new_student = {"educ_n":4,"age_n":0,"imd_n":90,"disab_n":0,
               "num_of_prev_attempts":0,"studied_credits":60}
print("\nNEW student (profile only) -> ", cold_start_recommend(new_student))
print("Total unknown (popularity)  -> ", popularity_recommend())

## Comparison & write-up notes (for the report)

**Content-based (A)**: recommends on course attributes + student profile.
*Strengths:* handles cold-start, every course is recommendable, explainable
("this course suits your profile"). *Weakness:* limited by how rich the metadata
is, and OULAD's course metadata is thin.

**Collaborative filtering (B)**: recommends on behaviour overlap. *Strengths:*
captures real patterns no metadata can; tends to win on hit-rate where co-
enrolment signal exists. *Weakness:* cold-start blind, and biased toward popular
courses.

**Verdict:** use **B when history exists, A for cold-start**, a standard hybrid.
Report the hit-rate of both *against the popularity baseline*, the coverage gap
(personalised methods recommend all 7; popularity repeats the same few), and the
7-course caveat so the evaluation reads honestly.